# Лабораторна робота №4: Візуалізація даних 2
**Тема:** Робота з інтерактивними інтерфейсами, генерація зашумлених сигналів та їх фільтрація.

## Інструкція користувача
Ця програма дозволяє моделювати графік функції гармоніки з накладеним шумом та фільтрувати його. 
* Використовуйте вкладку **"Гармоніка"** для налаштування амплітуди, частоти та фази основного сигналу.
* У вкладці **"Шум"** можна змінити математичне очікування та дисперсію шуму, а також повністю вимкнути його відображення. Зміна параметрів гармоніки не генерує новий шум.
* У вкладці **"Фільтр"** можна налаштувати параметри IIR-фільтра Баттерворта (порядок та частоту зрізу) для наближення результату до чистого сигналу.
* Кнопка **"Reset"** скидає всі повзунки до стандартних значень.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from scipy.signal import iirfilter, filtfilt

state = {'noise': None, 'mean': None, 'cov': None}
t = np.linspace(0, 10, 1000)

def harmonic_with_noise(amplitude, frequency, phase, noise_mean, noise_covariance, show_noise):
    harmonic = amplitude * np.sin(frequency * t + phase)
    
    if state['noise'] is None or state['mean'] != noise_mean or state['cov'] != noise_covariance:
        state['noise'] = np.random.normal(noise_mean, np.sqrt(noise_covariance), len(t))
        state['mean'] = noise_mean
        state['cov'] = noise_covariance
        
    if show_noise:
        return harmonic + state['noise']
    return harmonic

def update_plot(amplitude, frequency, phase, noise_mean, noise_covariance, show_noise, filter_order, cutoff):
    y_raw = harmonic_with_noise(amplitude, frequency, phase, noise_mean, noise_covariance, show_noise)
    y_clean = amplitude * np.sin(frequency * t + phase)
    
    b, a = iirfilter(filter_order, cutoff, btype='low', ftype='butter')
    y_filtered = filtfilt(b, a, y_raw)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    ax1.plot(t, y_raw, label='Сигнал із шумом' if show_noise else 'Чиста гармоніка', color='steelblue')
    if show_noise:
        ax1.plot(t, y_clean, label='Чиста гармоніка', color='black', linestyle='--', alpha=0.6)
    ax1.set_title('Початковий сигнал')
    ax1.grid(True, linestyle=':')
    ax1.legend()
    
    ax2.plot(t, y_clean, label='Оригінальна гармоніка', color='black', linestyle='--', alpha=0.6)
    ax2.plot(t, y_filtered, label='Відфільтрований сигнал', color='crimson')
    ax2.set_title('Результат фільтрації')
    ax2.grid(True, linestyle=':')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

w_amp = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Амплітуда:')
w_freq = widgets.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.1, description='Частота:')
w_phase = widgets.FloatSlider(value=0.0, min=0.0, max=np.pi*2, step=0.1, description='Фаза:')

w_mean = widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1, description='Середнє (μ):')
w_cov = widgets.FloatSlider(value=0.5, min=0.0, max=3.0, step=0.1, description='Дисперсія (σ²):')
w_show = widgets.Checkbox(value=True, description='Показувати шум')

w_order = widgets.IntSlider(value=4, min=1, max=8, description='Порядок:')
w_cutoff = widgets.FloatSlider(value=0.15, min=0.01, max=0.5, step=0.01, description='Частота зрізу:')

w_reset = widgets.Button(description='Reset', button_style='danger', icon='refresh')

def reset_params(b):
    w_amp.value = 1.0
    w_freq.value = 2.0
    w_phase.value = 0.0
    w_mean.value = 0.0
    w_cov.value = 0.5
    w_show.value = True
    w_order.value = 4
    w_cutoff.value = 0.15

w_reset.on_click(reset_params)

out = widgets.interactive_output(update_plot, {
    'amplitude': w_amp, 'frequency': w_freq, 'phase': w_phase,
    'noise_mean': w_mean, 'noise_covariance': w_cov, 'show_noise': w_show,
    'filter_order': w_order, 'cutoff': w_cutoff
})

tab = widgets.Tab()
tab.children = [
    widgets.VBox([w_amp, w_freq, w_phase]),
    widgets.VBox([w_mean, w_cov, w_show]),
    widgets.VBox([w_order, w_cutoff])
]
tab.set_title(0, 'Гармоніка')
tab.set_title(1, 'Шум')
tab.set_title(2, 'Фільтр')

ui = widgets.VBox([tab, w_reset, out])
display(ui)